In [1]:
import os 
os.chdir('../')

In [2]:
from pathlib import Path

import pandas as pd 
import numpy as np
import openmatrix as omx

from src.data.EDA.mtc_od import read_mtc_skims 

In [3]:
outpath = Path("data/interim/trip_generation")
outpath.mkdir(parents=True, exist_ok=True)

# SW Trip Generation 

In [13]:
path = "data/interim/matrix_projection/SwTAZ_to_TMTaz_TRIPS_FFM_2020.omx"
sw_od_omx = omx.open_file(path, 'r')

In [14]:
n = 1454
zone_ids = range(1, n + 1)
trips = pd.DataFrame(index=zone_ids)
trips.index.name = "TAZ1454"
# trips = trips.reset_index()

od_omx = omx.open_file(path, "r")
for name in od_omx.list_matrices():
    matrix = np.array(od_omx[name])
    mtc_matrix = matrix[:n, :n]
    trips[f"{name}_production"] = mtc_matrix.sum(axis = 1)
    trips[f"{name}_attraction"] = mtc_matrix.sum(axis = 1)
od_omx.close()
print(f"Total truck trips: {trips.sum().sum():,.0f}")

Total truck trips: 1,056,374


In [15]:
a = {
    "light_trucks_production": {"starts": ("LT"), "ends": ("production")}, 
    "light_trucks_attraction": {"starts": ("LT"), "ends": ("attraction")}, 
    "medium_trucks_production": {"starts": ("MT1", "MT2"), "ends": ("production")}, 
    "medium_trucks_attraction": {"starts": ("MT1", "MT2"), "ends": ("attraction")}, 
    "heavy_trucks_production": {"starts": ("HT"), "ends": ("production")}, 
    "heavy_trucks_attraction": {"starts": ("HT"), "ends": ("attraction")}, 
    
}
for col_name, attr in a.items():
    starts = attr["starts"]
    end = attr["ends"] 
    cols = [c for c in trips.columns if c.startswith(starts) and c.endswith(end)]
    trips[col_name] = trips[cols].sum(axis = 1)

In [16]:
val = trips[list(a.keys())].sum().sum()
print(f"Total truck trips: {val:,.0f}")

Total truck trips: 1,056,374


In [18]:
trips.to_parquet(Path(outpath, "SW_trip_generation_TAZ1454.parquet"))

# TM1.6 Trip Attraction

In [21]:
SKIMS_PATH = "data/interim/cube_io/mtc_skims/COM_HWYSKIM{tod}.omx"
skims = read_mtc_skims(SKIMS_PATH)

TRUCK_DISTRIB_LOS_TOLL_PART = 0.5

skims["blend_time_vstruck"] = (
    (([1/3] * skims["TIMEVSM_AM"] + [2/3] * skims["TIMEVSM_MD"]) * (1 - TRUCK_DISTRIB_LOS_TOLL_PART))
    + (([1/3] * skims["TOLLTIMEVSM_AM"] + [2/3] * skims["TOLLTIMEVSM_MD"]) * TRUCK_DISTRIB_LOS_TOLL_PART)
)
skims["blend_time_struck"] = (
    (([1/3] * skims["TIMESML_AM"] + [2/3] * skims["TIMESML_MD"]) * (1 - TRUCK_DISTRIB_LOS_TOLL_PART))
    + (([1/3] * skims["TOLLTIMESML_AM"] + [2/3] * skims["TOLLTIMESML_MD"]) * TRUCK_DISTRIB_LOS_TOLL_PART)
) 
skims["blend_time_medium"] = (
    (([1/3] * skims["TIMEMED_AM"] + [2/3] * skims["TIMEMED_MD"]) * (1 - TRUCK_DISTRIB_LOS_TOLL_PART))
    + (([1/3] * skims["TOLLTIMEMED_AM"] + [2/3] * skims["TOLLTIMEMED_MD"]) * TRUCK_DISTRIB_LOS_TOLL_PART)
) 
skims["blend_time_large"] = (
    (([1/3] * skims["TIMELRG_AM"] + [2/3] * skims["TIMELRG_MD"]) * (1 - TRUCK_DISTRIB_LOS_TOLL_PART))
    + (([1/3] * skims["TOLLTIMELRG_AM"] + [2/3] * skims["TOLLTIMELRG_MD"]) * TRUCK_DISTRIB_LOS_TOLL_PART)
)

In [23]:
skims.to_parquet(Path(outpath, "tm16_blended_skims.parquet"))

In [24]:
path = "data/interim/eda/SW_trip_distributions.parquet"
df = pd.read_parquet(path)

In [27]:
df[df["series"] == "light_trucks"]

,plot_id,source,series,x_col,bins,bin_id,bin_start,bin_end,center,trips,share
0,sw_distance,CSF2TDM,light_trucks,dist_comp,25,0,0.030000,6.833600,3.431800,112196.6826,2.820998e-01
1,sw_distance,CSF2TDM,light_trucks,dist_comp,25,1,6.833600,13.637200,10.235400,97208.1780,2.444137e-01
2,sw_distance,CSF2TDM,light_trucks,dist_comp,25,2,13.637200,20.440800,17.039000,73788.9273,1.855299e-01
3,sw_distance,CSF2TDM,light_trucks,dist_comp,25,3,20.440800,27.244400,23.842600,48415.4480,1.217326e-01
4,sw_distance,CSF2TDM,light_trucks,dist_comp,25,4,27.244400,34.048000,30.646200,29891.8068,7.515796e-02
5,sw_distance,CSF2TDM,light_trucks,dist_comp,25,5,34.048000,40.851600,37.449800,17648.5455,4.437432e-02
6,sw_distance,CSF2TDM,light_trucks,dist_comp,25,6,40.851600,47.655200,44.253400,9341.1801,2.348684e-02
7,sw_distance,CSF2TDM,light_trucks,dist_comp,25,7,47.655200,54.458800,51.057000,4877.3018,1.226316e-02
8,sw_distance,CSF2TDM,light_trucks,dist_comp,25,8,54.458800,61.262400,57.860600,2421.6474,6.088828e-03
9,sw_distance,CSF2TDM,light_trucks,dist_comp,25,9,61.262400,68.066000,64.664200,1079.1847,2.713430e-03
